# 실험 결과 비교, 평균 ± 표준편차, paired t-test

In [9]:
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

STEP = 10  # 비교에 쓸 샘플링 스텝 수
CLASS_NAMES = {"0": "Normal", "1": "Underfill_50FR", "2": "Underfill_Fan", "macro": "macro"}

SOURCES = {  # 라벨 → (CSV 경로, 이 CSV에서 쓸 method 정규식)
    "exp0_baseline": ("exp0_baseline/exp0_baseline.csv", r"^0_baseline"),
    "exp1_oversampling": ("exp1_oversampling_step_10/exp1_oversampling.csv",
                          rf"^1_diffusion_oversampling_ddpm1000_step{STEP}$"),
    "exp3_filtered_oversampling": ("exp3_filtered_oversampling/exp3_filtered_oversampling.csv",
                                   rf"^3_filtered_oversampling_ddpm1000_step{STEP}$"),
    "exp4_filtered_joint": ("exp4_filtered_joint/exp4_joint_diffusion_step10.csv",
                            r"^4_joint_diffusion_ddpm1000_step\d+_filtered$"),
}

frames = []
for label, (path, pat) in SOURCES.items():
    if not os.path.exists(path):
        print("결과 CSV 없음, 건너뜀:", path); continue
    d = pd.read_csv(path)
    d = d[d["method"].str.contains(pat, regex=True)].assign(method=label)
    frames.append(d)

df_all = pd.concat(frames, ignore_index=True)
df_all["class"] = df_all["class"].astype(str)
df_all = df_all.drop_duplicates(subset=["method", "seed", "class"], keep="last")  # 이어서 돌려 중복된 행은 마지막 것만
COMMON_SEEDS = sorted(df_all[(df_all["method"] == "exp4_filtered_joint") & (df_all["class"] == "macro")]["seed"])
df_all = df_all[df_all["seed"].isin(COMMON_SEEDS)]  # 필터 joint(exp4)가 돈 seed로만 모든 실험을 비교한다
print("비교에 쓴 seed:", COMMON_SEEDS)
METHOD_ORDER = [m for m in SOURCES if m in set(df_all["method"])]
print("실험별 seed 수:")
print(df_all[df_all["class"] == "macro"].groupby("method")["seed"].nunique().reindex(METHOD_ORDER).to_string())

비교에 쓴 seed: [0, 3, 4, 5, 7, 8, 9]
실험별 seed 수:
method
exp0_baseline                 7
exp1_oversampling             7
exp3_filtered_oversampling    7
exp4_filtered_joint           7


## 1. 실험별 평균 ± 표준편차

In [10]:
def per_seed(metric, cls="macro"):
    sub = df_all[df_all["class"] == str(cls)]
    return sub.pivot_table(index="seed", columns="method", values=metric, aggfunc="mean")[METHOD_ORDER]


summary_items = [("accuracy", "macro"), ("precision", "macro"), ("recall", "macro"), ("f1", "macro")]
summary = {}
for metric, cls in summary_items:
    tab = per_seed(metric, cls)
    name = f"{metric}" if cls == "macro" else f"{metric}_{CLASS_NAMES[cls]}"
    summary[name] = tab.mean().map("{:.3f}".format) + " ± " + tab.std(ddof=1).map("{:.3f}".format)
summary = pd.DataFrame(summary)
summary.insert(0, "n_seeds", per_seed("f1").notna().sum())
summary

,n_seeds,accuracy,precision,recall,f1
method,,,,,
exp0_baseline,7,0.764 ± 0.032,0.823 ± 0.020,0.764 ± 0.032,0.740 ± 0.046
exp1_oversampling,7,0.771 ± 0.066,0.820 ± 0.037,0.771 ± 0.066,0.750 ± 0.086
exp3_filtered_oversampling,7,0.832 ± 0.051,0.858 ± 0.033,0.832 ± 0.051,0.825 ± 0.058
exp4_filtered_joint,7,0.770 ± 0.037,0.829 ± 0.030,0.770 ± 0.037,0.752 ± 0.048


## 2. paired t-test

In [11]:
COMPARISONS = [
    ("exp1_oversampling", "exp0_baseline", "오버샘플링 vs baseline"),
    ("exp3_filtered_oversampling", "exp1_oversampling", "필터 오버샘플링 vs 오버샘플링"),
    ("exp3_filtered_oversampling", "exp0_baseline", "필터 오버샘플링 vs baseline"),
    ("exp4_filtered_joint", "exp1_oversampling", "필터 joint vs 오버샘플링"),
    ("exp4_filtered_joint", "exp3_filtered_oversampling", "필터 joint vs 필터 오버샘플링"),
    ("exp4_filtered_joint", "exp0_baseline", "필터 joint vs baseline"),
]
COMPARISONS = [c for c in COMPARISONS if c[0] in METHOD_ORDER and c[1] in METHOD_ORDER]


def paired_tests(metric="f1", cls="macro", comparisons=COMPARISONS, df=None):
    df = df_all if df is None else df
    sub = df[df["class"] == str(cls)]
    table = sub.pivot_table(index="seed", columns="method", values=metric, aggfunc="mean")
    rows = []
    for a, b, desc in comparisons:
        if a not in table or b not in table:
            continue
        pair = table[[a, b]].dropna()
        n = len(pair)
        diff = pair[a] - pair[b]
        row = {"비교": desc, "A": a, "B": b, "n": n,
               "mean_A": pair[a].mean(), "mean_B": pair[b].mean()}
        if n >= 2 and diff.std(ddof=1) > 0:
            t_stat, p_val = stats.ttest_rel(pair[a], pair[b])
            row.update({"t": t_stat, "p": p_val})
        else:
            row.update({"t": np.nan, "p": np.nan})
        rows.append(row)
    res = pd.DataFrame(rows)
    if res.empty:
        print(f"[{metric}/{cls}] 비교할 수 있는 실험 쌍이 아직 없습니다.")
        return res
    res["유의(p<0.05)"] = res["p"] < 0.05
    return res


pd.set_option("display.float_format", "{:.4f}".format)
paired_tests("f1", "macro")

,비교,A,B,n,mean_A,mean_B,t,p,유의(p<0.05)
0,오버샘플링 vs baseline,exp1_oversampling,exp0_baseline,7,0.7501,0.7398,0.2819,0.7875,False
1,필터 오버샘플링 vs 오버샘플링,exp3_filtered_oversampling,exp1_oversampling,7,0.8253,0.7501,2.0084,0.0914,False
2,필터 오버샘플링 vs baseline,exp3_filtered_oversampling,exp0_baseline,7,0.8253,0.7398,3.5913,0.0115,True
3,필터 joint vs 오버샘플링,exp4_filtered_joint,exp1_oversampling,7,0.7517,0.7501,0.0818,0.9375,False
4,필터 joint vs 필터 오버샘플링,exp4_filtered_joint,exp3_filtered_oversampling,7,0.7517,0.8253,-2.8589,0.0288,True
5,필터 joint vs baseline,exp4_filtered_joint,exp0_baseline,7,0.7517,0.7398,0.6648,0.5309,False
